# Day 079 — Exercise 2: Parsing LLM Output

**What you'll build:** a tolerant JSON parser and an action parser that never raises.

**Why it matters:** llama3.2 won't always return perfect JSON — it wraps output in ```` ```json ```` fences or adds a sentence first. The parser has to cope, and `parse_action` must always return *something* so the loop can't crash on a malformed reply.

In [ ]:
import json

def _make_mock_llm(script):
    """Return an llm_fn(messages) that yields each scripted reply in turn.

    Repeats the last reply once the script is exhausted - handy for testing
    a runaway loop (a model that never says 'finish').
    """
    state = {'i': 0}
    def _fn(messages):
        i = state['i']
        state['i'] = min(i + 1, len(script) - 1)
        return script[i]
    return _fn
import ast
import json
import operator

# ── a safe calculator tool (no eval) ─────────────────────────────────────────
_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,
    ast.USub: operator.neg, ast.UAdd: operator.pos,
}


def _eval_node(node):
    """Recursively evaluate an arithmetic AST node. Raises on anything unsafe."""
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.left), _eval_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.operand))
    raise ValueError("unsupported expression")


def safe_calculate(expression):
    """Evaluate a basic arithmetic expression without eval().

    Supports + - * / ** % and parentheses. Anything else (names, calls,
    attribute access) raises ValueError. This is the safe way to give an
    agent a calculator: never eval() untrusted model output.
    """
    tree = ast.parse(expression, mode="eval")
    return _eval_node(tree.body)


# ── the tool registry ────────────────────────────────────────────────────────
# A tool = {description, parameters, fn}. fn takes an args dict, returns a str.
DEFAULT_TOOLS = {
    "calculator": {
        "description": "Evaluate an arithmetic expression, e.g. 2 * (3 + 4).",
        "parameters": {"expression": "string - the arithmetic to evaluate"},
        "fn": lambda args: str(safe_calculate(args["expression"])),
    },
    "word_count": {
        "description": "Count the words in a piece of text.",
        "parameters": {"text": "string - the text to count words in"},
        "fn": lambda args: str(len(str(args["text"]).split())),
    },
}


def build_tool_descriptions(tools):
    """Render a tool registry as a text block for the prompt."""
    lines = []
    for name, spec in tools.items():
        params = ", ".join(spec.get("parameters", {}))
        lines.append("- " + name + "(" + params + "): " + spec["description"])
    return "\n".join(lines)


## Task

1. `safe_parse_json(text) -> dict | None` — slice from the first `{` to the last `}` (`text.find('{')`, `text.rfind('}')`), `json.loads` that slice inside `try/except`. Return the dict, or `None` if there's no object or it doesn't parse.
2. `parse_action(text) -> dict` — call `safe_parse_json`. If it's a dict with a `tool` that isn't `'finish'`: return `{'type':'tool','tool':..,'args':..}`. Otherwise return `{'type':'finish','answer':..}` (use the parsed `answer`, or the raw text). Must **never** raise.

## Your Implementation

In [ ]:
def safe_parse_json(text):
    """Extract + parse the first JSON object from messy text. Returns dict|None."""
    raise NotImplementedError

def parse_action(text):
    """Turn LLM text into an action dict. NEVER raises.
    {'type':'tool','tool':..,'args':..} or {'type':'finish','answer':..}.
    """
    raise NotImplementedError


In [ ]:

# ── parsing messy LLM output ──────────────────────────────────────────────────
def safe_parse_json(text):
    """Extract and parse the first JSON object from messy LLM output.

    LLMs wrap JSON in markdown fences or prose. Instead of fighting that,
    slice from the first '{' to the last '}' and parse that. Returns a dict,
    or None if no valid JSON object is present.
    """
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1 or end < start:
        return None
    try:
        data = json.loads(text[start:end + 1])
    except (json.JSONDecodeError, ValueError):
        return None
    return data if isinstance(data, dict) else None


def parse_action(text):
    """Turn raw LLM output into an action dict. NEVER raises.

    Returns one of:
      {"type": "tool",   "tool": name, "args": {...}}
      {"type": "finish", "answer": str}
    If the text is not a valid tool call, it falls back to a finish action
    holding the raw text - so a badly-formatted model reply still terminates
    the loop instead of crashing it.
    """
    data = safe_parse_json(text)
    if not isinstance(data, dict):
        return {"type": "finish", "answer": text.strip()}
    tool = data.get("tool")
    if tool and tool != "finish":
        return {"type": "tool", "tool": tool, "args": data.get("args", {})}
    return {"type": "finish", "answer": data.get("answer", text.strip())}


## Automated checks

In [ ]:

score, total = 0, 5
try:
    clean = safe_parse_json('{"tool": "calculator", "args": {"expression": "2+2"}}')
    assert clean['tool'] == 'calculator'
    score += 1; print("✅ safe_parse_json parses clean JSON")

    fenced = 'Sure!\n```json\n{"tool": "finish", "answer": "hi"}\n```'
    assert safe_parse_json(fenced)['answer'] == 'hi'
    score += 1; print("✅ safe_parse_json tolerates fences and prose")

    assert safe_parse_json('no json here at all') is None
    score += 1; print("✅ safe_parse_json returns None on garbage")

    a = parse_action('{"tool": "calculator", "args": {"expression": "1+1"}}')
    assert a['type'] == 'tool' and a['tool'] == 'calculator'
    assert a['args']['expression'] == '1+1'
    score += 1; print("✅ parse_action extracts a tool action")

    f = parse_action('The answer is 42, no JSON here.')
    assert f['type'] == 'finish' and '42' in f['answer']
    score += 1; print("✅ parse_action falls back to finish (never raises)")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python

# ── parsing messy LLM output ──────────────────────────────────────────────────
def safe_parse_json(text):
    """Extract and parse the first JSON object from messy LLM output.

    LLMs wrap JSON in markdown fences or prose. Instead of fighting that,
    slice from the first '{' to the last '}' and parse that. Returns a dict,
    or None if no valid JSON object is present.
    """
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1 or end < start:
        return None
    try:
        data = json.loads(text[start:end + 1])
    except (json.JSONDecodeError, ValueError):
        return None
    return data if isinstance(data, dict) else None


def parse_action(text):
    """Turn raw LLM output into an action dict. NEVER raises.

    Returns one of:
      {"type": "tool",   "tool": name, "args": {...}}
      {"type": "finish", "answer": str}
    If the text is not a valid tool call, it falls back to a finish action
    holding the raw text - so a badly-formatted model reply still terminates
    the loop instead of crashing it.
    """
    data = safe_parse_json(text)
    if not isinstance(data, dict):
        return {"type": "finish", "answer": text.strip()}
    tool = data.get("tool")
    if tool and tool != "finish":
        return {"type": "tool", "tool": tool, "args": data.get("args", {})}
    return {"type": "finish", "answer": data.get("answer", text.strip())}
```

**Why slice the braces instead of a regex?** The first `{` to the last `}` is the JSON object regardless of any fences or prose around it — one line, no regex, and it degrades to `None` cleanly when there's no object.

</details>